# Dijkstra Algorithm - Nearest Hospitals by Travel Time

**Input:** patient latitude + longitude (the 3 nearest hospitals are always returned).  
**Output:** rank, phones, hospital_id, name, governorate, address, travel_time_min (whole minutes).  
**Uses all hospitals** in the CSV (no filtering by maternity tier).

Flow: load hospitals -> road graph -> snap hospitals/patient to road nodes -> Dijkstra -> top 3.

> The road graph used when you run the last cell is a **temporary stand-in** (straight-line x 1.3 at 40 km/h).
> Replace it with the real road network (`load_osm_graph([...])` or the teammate's graph) for realistic times.

In [ ]:
import heapq
import itertools
import math
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

TOP_K = 3
DEFAULT_SPEED_KMH = 30.0
# None = use ALL hospitals in the CSV (no filtering by maternity tier).
# To filter later, e.g.: ["T1_maternity_hospital", "T2_general_hospital_unverified"]
ALLOWED_TIERS = None
HOSPITALS_CSV = "final_maternal_facilities_clean.csv"

Graph = Dict[Any, List[Tuple[Any, dict]]]

Put `final_maternal_facilities_clean.csv` in the same folder as this notebook (or change `HOSPITALS_CSV` above).

## 1. data

In [ ]:
# ---------------------------------------------------------------- 1. data
def load_and_validate_hospitals(csv_path: str, allowed_tiers: Optional[List[str]] = None):
    """Load the CSV, check columns, drop rows without coordinates, keep only allowed tiers."""
    df = pd.read_csv(csv_path)
    required = ["canonical_entity_id", "canonical_name", "latitude_final",
                "longitude_final", "birth_service_tier"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Columns missing from CSV: {missing}")

    df = df.dropna(subset=["latitude_final", "longitude_final", "canonical_entity_id"]).copy()
    df["canonical_entity_id"] = df["canonical_entity_id"].astype(str)

    if allowed_tiers is not None:
        before = len(df)
        df = df[df["birth_service_tier"].isin(allowed_tiers)].copy()
        print(f"Filtered by birth_service_tier: {before} -> {len(df)} hospitals")
    if df.empty:
        raise ValueError("No valid hospitals after filtering. Check birth_service_tier values.")
    return df, df["canonical_entity_id"].tolist()

## 2. edge cost

In [ ]:
# ---------------------------------------------------------------- 2. edge cost
def edge_cost(u, v, edge_data: dict) -> float:
    """Cost of one road segment = travel time in minutes = length / speed * 60."""
    length_km = float(edge_data.get("length_km", 0.0))
    speed_kmh = float(edge_data.get("speed_kmh", DEFAULT_SPEED_KMH))
    if speed_kmh <= 0:
        speed_kmh = DEFAULT_SPEED_KMH
    return length_km / speed_kmh * 60.0

## 3. graph

In [ ]:
# ---------------------------------------------------------------- 3. graph
def build_graph(edges, bidirectional: bool = True) -> Graph:
    """
    edges: iterable of (u, v, length_km) or (u, v, length_km, speed_kmh).
    Roads are two-way by default; pass bidirectional=False for one-way data.
    """
    graph: Graph = {}
    for e in edges:
        u, v, length_km = e[0], e[1], float(e[2])
        data = {"length_km": length_km}
        if len(e) > 3 and e[3] is not None:
            data["speed_kmh"] = float(e[3])
        graph.setdefault(u, []).append((v, dict(data)))
        graph.setdefault(v, [])
        if bidirectional:
            graph[v].append((u, dict(data)))
    return graph


def haversine_km(lat1, lon1, lat2, lon2) -> float:
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def nearest_node(lat: float, lon: float, node_coords: Dict[Any, Tuple[float, float]]):
    """Snap a point to the closest graph node. node_coords: {node: (lat, lon)}."""
    return min(node_coords, key=lambda n: haversine_km(lat, lon, *node_coords[n]))


def snap_hospitals_to_nodes(hospitals_df: pd.DataFrame, node_coords) -> Dict[str, Any]:
    """{hospital_id: nearest road-graph node}. Hospital IDs and graph nodes are DIFFERENT things."""
    return {
        str(r.canonical_entity_id): nearest_node(r.latitude_final, r.longitude_final, node_coords)
        for r in hospitals_df.itertuples()
    }

## 4. Dijkstra

In [ ]:
# ---------------------------------------------------------------- 4. Dijkstra
def dijkstra(graph: Graph, source, cost_fn=edge_cost, targets=None):
    """
    Returns (dist, prev). dist = minutes from source, prev = predecessor for path rebuilding.
    If `targets` is given, stops as soon as all of them are settled (faster on big graphs).
    """
    dist = {source: 0.0}
    prev = {source: None}
    counter = itertools.count()          # tie-breaker: heap never compares nodes (str vs int crash)
    pq = [(0.0, next(counter), source)]
    remaining = set(targets) if targets else None

    while pq:
        cost, _, u = heapq.heappop(pq)
        if cost > dist.get(u, math.inf):
            continue
        if remaining is not None:
            remaining.discard(u)
            if not remaining:
                break
        for v, data in graph.get(u, []):
            w = cost_fn(u, v, data)
            if w < 0:
                raise ValueError("Dijkstra does not allow negative edge costs.")
            nc = cost + w
            if nc < dist.get(v, math.inf):
                dist[v] = nc
                prev[v] = u
                heapq.heappush(pq, (nc, next(counter), v))
    return dist, prev


def reconstruct_path(prev: dict, target) -> list:
    path, node = [], target
    while node is not None:
        path.append(node)
        node = prev.get(node)
    return path[::-1]

## 5. top-k hospitals

In [ ]:
# ---------------------------------------------------------------- 5. top-k hospitals
def find_nearest_hospitals(graph: Graph, source, hospital_to_node: Dict[str, Any],
                           top_k: int = TOP_K, cost_fn=edge_cost) -> dict:
    """
    graph            : road graph
    source           : graph node of the patient
    hospital_to_node : {hospital_id: graph_node}  (ONLY already-filtered maternity hospitals)
    """
    if not hospital_to_node:
        raise ValueError("hospital_to_node is empty.")
    if source not in graph:
        raise ValueError(f"Source node {source!r} is not in the graph.")

    dist, prev = dijkstra(graph, source, cost_fn, targets=set(hospital_to_node.values()))

    cands, unreachable = [], []
    for hid, node in hospital_to_node.items():
        if node in dist:
            cands.append((dist[node], str(hid), node))
        else:
            unreachable.append(str(hid))
    if unreachable:
        print(f"{len(unreachable)} hospital(s) not reachable from source (e.g. {unreachable[:3]})")
    if not cands:
        raise ValueError("No reachable hospitals from the given source.")

    cands.sort(key=lambda x: (x[0], x[1]))          # time, then id for stable ties
    results = [{"rank": i, "hospital_id": hid, "travel_time_min": round(t, 2),
                "path": reconstruct_path(prev, node)}
               for i, (t, hid, node) in enumerate(cands[:top_k], 1)]
    return {"best_option": results[0], "top_k": results, "unreachable": unreachable}

## 6. road graph sources

In [ ]:
# ================================================================ 6. road graph sources
def graph_from_networkx(G, default_speed_kmh: float = DEFAULT_SPEED_KMH):
    """
    Convert an OSMnx / networkx road graph into (graph, node_coords).
    Expects node attributes x (lon), y (lat) and edge attribute length (metres).
    Uses edge attribute speed_kph when present (osmnx.add_edge_speeds), else default speed.
    Keeps the fastest parallel edge between two nodes.
    """
    node_coords = {n: (d["y"], d["x"]) for n, d in G.nodes(data=True)}
    best = {}
    for u, v, d in G.edges(data=True):
        length_km = float(d.get("length", 0.0)) / 1000.0
        speed = float(d.get("speed_kph", default_speed_kmh) or default_speed_kmh)
        t = length_km / speed
        if (u, v) not in best or t < best[(u, v)][0]:
            best[(u, v)] = (t, length_km, speed)
    graph: Graph = {n: [] for n in G.nodes}
    for (u, v), (_, l, sp) in best.items():
        graph[u].append((v, {"length_km": l, "speed_kmh": sp}))
    return graph, node_coords


def load_osm_graph(place_names: List[str]):
    """
    Real road network from OpenStreetMap (needs `pip install osmnx` + internet), e.g.
        graph, node_coords = load_osm_graph(["Sohag Governorate, Egypt", "Assiut Governorate, Egypt"])
    """
    import osmnx as ox
    G = ox.graph_from_place(place_names, network_type="drive")
    G = ox.add_edge_speeds(G)
    return graph_from_networkx(G)


def build_toy_graph(points: Dict[Any, Tuple[float, float]], k: int = 5,
                    detour: float = 1.3, speed_kmh: float = 40.0):
    """
    TEST-ONLY stand-in for a road network: links each point to its k nearest points and joins
    all parts with a minimum spanning tree so everything is reachable. Distance = straight line
    x detour factor. Replace with load_osm_graph() / the teammate's real graph.
    """
    ids = list(points)
    edges, in_tree, link = [], {ids[0]}, {}
    for i in ids:
        near = sorted((haversine_km(*points[i], *points[j]), j) for j in ids if j != i)[:k]
        edges += [(i, j, d * detour, speed_kmh) for d, j in near]
    best = {j: (haversine_km(*points[ids[0]], *points[j]), ids[0]) for j in ids[1:]}
    while best:                                     # Prim: connect the separate clusters
        j = min(best, key=lambda x: best[x][0]); d, i = best.pop(j)
        edges.append((i, j, d * detour, speed_kmh)); in_tree.add(j)
        for m in best:
            dm = haversine_km(*points[j], *points[m])
            if dm < best[m][0]:
                best[m] = (dm, j)
    return build_graph(edges), dict(points)

## 7. one-call entry point

In [ ]:
# ================================================================ 7. one-call entry point
def recommend_hospitals(patient_lat: float, patient_lon: float, hospitals_df: pd.DataFrame,
                        graph: Graph, node_coords, top_k: int = TOP_K,
                        one_per_location: bool = True) -> pd.DataFrame:
    """
    INPUTS : patient latitude + longitude, hospitals table, road graph (+ its node coordinates).
    OUTPUT : table of the nearest hospitals by travel time (all hospitals, no specialty filter).
    one_per_location=True: hospitals at the same road node count as ONE option (the others are listed in
    `same_location_ids`), so the top-k are k different places.
    """
    h2n = snap_hospitals_to_nodes(hospitals_df, node_coords)
    src = nearest_node(patient_lat, patient_lon, node_coords)
    big = len(h2n)
    res = find_nearest_hospitals(graph, src, h2n, top_k=big)["top_k"]      # ranked, all reachable

    info = hospitals_df.set_index("canonical_entity_id")

    def val(hid, col):                      # value from the CSV; empty string if the column/value is missing
        if col not in info.columns or pd.isna(info.loc[hid, col]):
            return ""
        return info.loc[hid, col]

    rows, seen = [], {}
    for r in res:
        node = r["path"][-1]
        if one_per_location and node in seen:
            seen[node]["same_location_ids"].append(r["hospital_id"])
            continue
        row = {"rank": len(rows) + 1, "hospital_id": r["hospital_id"],
               "name": info.loc[r["hospital_id"], "canonical_name"],
               "governorate": info.loc[r["hospital_id"], "governorate_assigned"],
               "district": info.loc[r["hospital_id"], "district"],
               "address": val(r["hospital_id"], "address"),
               "phones_normalized_all": val(r["hospital_id"], "phones_normalized_all"),
               "phone": val(r["hospital_id"], "phone"),
               "tier": info.loc[r["hospital_id"], "birth_service_tier"],
               "travel_time_min": int(math.ceil(r["travel_time_min"] - 1e-9)), "path_nodes": len(r["path"]),
               "path": r["path"], "same_location_ids": []}
        rows.append(row); seen[node] = row
        if len(rows) == top_k:
            break
    return pd.DataFrame(rows)


def run_interactive(csv_path: str = HOSPITALS_CSV):
    """Asks ONLY for the patient's latitude and longitude, then prints the 3 nearest hospitals."""
    hospitals_df, _ = load_and_validate_hospitals(csv_path, ALLOWED_TIERS)
    pts = {i: p for i, p in enumerate(sorted({(round(r.latitude_final, 5), round(r.longitude_final, 5))
                                              for r in hospitals_df.itertuples()}))}
    graph, node_coords = build_toy_graph(pts)               # <- replace with load_osm_graph([...])
    lat = float(input("Patient latitude  (e.g. 26.5560): "))
    lon = float(input("Patient longitude (e.g. 31.6948): "))
    out = recommend_hospitals(lat, lon, hospitals_df, graph, node_coords, top_k=3)
    print()
    print(out[["rank", "phones_normalized_all", "hospital_id", "name", "governorate",
               "address", "travel_time_min"]].to_string(index=False))
    return out

## 8. Quick self-check (sample graph, answers calculated by hand)

In [ ]:
edges = [("S","A",2,60),("S","B",5,60),("A","B",1,60),("A","C",7,60),("B","C",3,60),
         ("B","H1",2,60),("C","H2",1,60),("A","H3",9,60),("X","Y",1,60)]   # X-Y is disconnected
G = build_graph(edges)
r = find_nearest_hospitals(G, "S", {"H1":"H1","H2":"H2","H3":"H3","HX":"Y"})["top_k"]
assert [(x["hospital_id"], x["travel_time_min"]) for x in r] == [("H1",5.0),("H2",7.0),("H3",11.0)]
assert r[0]["path"] == ["S","A","B","H1"]
print("Self-check passed:", [(x["hospital_id"], x["travel_time_min"], x["path"]) for x in r])

## 9. Run it - enter the patient's latitude and longitude

In [ ]:
result = run_interactive()